<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Notebooks/18_RNNs_and_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 18: Live Demos
## Embeddings, RNN Hidden States, and Attention Visualization

Three standalone demos to accompany the Session 18 slides:

1. **Demo 1: Embedding Arithmetic** (~5 min) — word vectors capture meaning
2. **Demo 2: RNN Hidden States** (~8 min) — watching memory evolve step by step
3. **Demo 3: Attention Visualization** (~10 min) — what does self-attention actually look at?

Each demo can be run independently. Run the Setup cell first, then jump to any demo.

---
# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
import os
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Base imports successful!")
print("Individual demos will install additional packages as needed.")

---
# Demo 1: Embedding Arithmetic

**Goal:** Show that word embeddings capture semantic meaning as geometry.

We'll load pretrained GloVe vectors (trained on billions of words) and demonstrate:
- Similar words have similar vectors
- Vector arithmetic encodes relationships: king - man + woman ≈ queen

### Download GloVe Vectors

GloVe 50d: 50-dimensional vectors for 400,000 words (~66MB download). Trained on 6 billion tokens from Wikipedia and web text.

In [ ]:
# Download GloVe 50d vectors (one-time, ~66MB)
import urllib.request
import zipfile

glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip = "glove.6B.zip"
glove_file = "glove.6B.50d.txt"

if not os.path.exists(glove_file):
    print("Downloading GloVe vectors (~66MB)...")
    urllib.request.urlretrieve(glove_url, glove_zip)
    print("Extracting 50d vectors...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract(glove_file)
    os.remove(glove_zip)
    print("Done!")
else:
    print("GloVe vectors already downloaded.")

In [ ]:
# Load GloVe into a dictionary
def load_glove(filepath):
    embeddings = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vector
    return embeddings

glove = load_glove(glove_file)
print(f"Loaded {len(glove):,} word vectors")
print(f"Each vector has {len(glove['the'])} dimensions")

### Word Similarity

If embeddings capture meaning, similar words should have similar vectors. We measure similarity with cosine similarity (same metric from Session 9: KNN).

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def most_similar(word, glove, top_n=5):
    """Find the most similar words to a given word."""
    if word not in glove:
        print(f"'{word}' not in vocabulary")
        return
    target = glove[word]
    similarities = []
    for w, vec in glove.items():
        if w != word:
            similarities.append((w, cosine_similarity(target, vec)))
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]



In [ ]:
# Try it
word = "spaghetti"
print(f"Words most similar to '{word}':")
for w, sim in most_similar(word, glove):
    print(f"  {w:15s} {sim:.4f}")

In [ ]:
# Compare: similar words vs unrelated words
pairs_similar = [("cat", "kitten"), ("happy", "joyful"), ("car", "automobile")]
pairs_different = [("cat", "democracy"), ("happy", "refrigerator"), ("car", "philosophy")]

print("Similar pairs:")
for w1, w2 in pairs_similar:
    sim = cosine_similarity(glove[w1], glove[w2])
    print(f"  {w1:15s} ↔ {w2:15s}  similarity: {sim:.4f}")

print(f"\nUnrelated pairs:")
for w1, w2 in pairs_different:
    sim = cosine_similarity(glove[w1], glove[w2])
    print(f"  {w1:15s} ↔ {w2:15s}  similarity: {sim:.4f}")

### Vector Arithmetic

The famous result: relationships between words are encoded as directions in vector space.

**king - man + woman ≈ ?**

In [ ]:
def analogy(a, b, c, glove, top_n=5):
    """
    Solve: a is to b as c is to ___
    Computes: b - a + c, finds nearest word
    Example: analogy('man', 'king', 'woman') -> 'queen'
    """
    if a not in glove or b not in glove or c not in glove:
        print(f"Word not in vocabulary")
        return

    # The arithmetic
    target = glove[b] - glove[a] + glove[c]

    # Find closest words (excluding the input words)
    exclude = {a, b, c}
    similarities = []
    for word, vec in glove.items():
        if word not in exclude:
            similarities.append((word, cosine_similarity(target, vec)))
    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_n]

# The classic example
print("king - man + woman = ?")
print("-" * 35)
results = analogy("man", "king", "woman", glove)
for word, sim in results:
    print(f"  {word:15s} {sim:.4f}")

In [ ]:
# More analogies
analogies = [
    ("man", "king", "woman",    "queen"),
    ("paris", "france", "rome", "italy"),
    ("slow", "slower", "fast",  "faster"),
    ("man", "doctor", "woman",  "nurse"),


print(f"{'Relationship':<35s} {'Expected':<10s} {'Top Result':<10s} {'Match?'}")
print("=" * 70)
for a, b, c, expected in analogies:
    results = analogy(a, b, c, glove)
    top_word = results[0][0]
    match = "✓" if top_word == expected else "✗"
    print(f"{a:8s} → {b:8s} as {c:8s} → ___    {expected:<10s} {top_word:<10s} {match}")

In [ ]:
# More analogies
analogies = [
    ("comedy", "horror", "talk",   ""),
    ("comedy", "horror", "jazz",   ""),
    ("soccer", "football", "brazil",   "")]

print(f"{'Relationship':<35s} {'Expected':<10s} {'Top Result':<10s} {'Match?'}")
print("=" * 70)
for a, b, c, expected in analogies:
    results = analogy(a, b, c, glove)
    top_word = results[0][0]
    match = "✓" if top_word == expected else "✗"
    print(f"{a:8s} → {b:8s} as {c:8s} → ___    {expected:<10s} {top_word:<10s} {match}")

### Visualizing Embedding Space

Let's project word vectors down to 2D with PCA (Session 10) and see if semantic clusters emerge.

In [ ]:
from sklearn.decomposition import PCA

# Pick word groups
word_groups = {
    'Animals': ['cat', 'dog', 'fish', 'bird', 'horse', 'bear', 'lion', 'wolf'],
    'Countries': ['france', 'germany', 'japan', 'brazil', 'italy', 'spain', 'china', 'india'],
    'Colors': ['red', 'blue', 'green', 'yellow', 'black', 'white', 'purple', 'orange'],
    'Other': ['coke', 'pepsi', 'soccer', 'football', 'basketball', 'vacuum', 'mop', 'sofa', 'space'],
}

# Collect vectors
all_words = []
all_vectors = []
all_labels = []
for group, words in word_groups.items():
    for w in words:
        if w in glove:
            all_words.append(w)
            all_vectors.append(glove[w])
            all_labels.append(group)

# PCA to 2D
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(np.array(all_vectors))

# Plot
colors = {'Animals': 'steelblue', 'Countries': 'coral', 'Colors': 'forestgreen', 'Other': 'red'}
fig, ax = plt.subplots(figsize=(12, 8))

for group in word_groups:
    mask = [l == group for l in all_labels]
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=colors[group], label=group, s=100,
               edgecolors='white', linewidth=0.5)

for i, word in enumerate(all_words):
    ax.annotate(word, (coords[i, 0], coords[i, 1]),
                fontsize=10, ha='center', va='bottom',
                xytext=(0, 6), textcoords='offset points')

ax.set_title('Word Embeddings Projected to 2D (PCA)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.1%}")

**Takeaways from Demo 1:**
- Word embeddings encode meaning as geometry: similar words cluster together
- Vector arithmetic captures relationships (king - man + woman ≈ queen)
- Embeddings also encode biases from training data (the doctor/nurse example)
- These pretrained vectors give models a head start on understanding language

---
# Demo 2: RNN Hidden States

**Goal:** See how an RNN's hidden state evolves as it reads a sequence, step by step.

We'll:
1. Build a quick text preprocessing pipeline
2. Train a small LSTM on real IMDB reviews
3. Visualize hidden states to see the "memory" in action

### Text Preprocessing Pipeline

Same pipeline from the slides: tokenize → build vocabulary → encode → pad.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
def simple_tokenize(text):
    """Lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()



class Vocabulary:
    def __init__(self, max_size=10000):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.max_size = max_size

    def build(self, tokenized_texts):
        """Build vocabulary from most common words."""
        counts = Counter()
        for tokens in tokenized_texts:
            counts.update(tokens)
        # Keep only top max_size - 2 words (reserve PAD and UNK)
        for word, _ in counts.most_common(self.max_size - 2):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def __len__(self):
        return len(self.word2idx)

    def encode(self, tokens):
        return [self.word2idx.get(t, 1) for t in tokens]

    def decode(self, indices):
        return [self.idx2word.get(i, '<UNK>') for i in indices]

def pad_sequences(encoded_texts, max_len):
    padded = []
    for seq in encoded_texts:
        if len(seq) > max_len:
            padded.append(seq[:max_len])
        else:
            padded.append(seq + [0] * (max_len - len(seq)))
    return np.array(padded)

print("Preprocessing utilities ready.")

### Load IMDB Reviews

We'll use a 5,000-review subset to keep training under 2 minutes.

In [ ]:
# Download IMDB dataset via keras (just the data, not the framework)
from tensorflow.keras.datasets import imdb as imdb_loader

# Load with a 10,000-word vocabulary limit
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = imdb_loader.load_data(num_words=10000)

# Get the word index to decode reviews back to text
word_index = imdb_loader.get_word_index()
# Keras offsets indices by 3 (0=pad, 1=start, 2=unknown, 3=unused)
idx_to_word = {v + 3: k for k, v in word_index.items()}
idx_to_word[0] = '<PAD>'
idx_to_word[1] = '<START>'
idx_to_word[2] = '<UNK>'

def decode_review(encoded_review):
    return ' '.join(idx_to_word.get(i, '?') for i in encoded_review)

# Show a sample
print("Sample review (first 100 words):")
print(decode_review(X_train_raw[0])[:500])
print(f"\nLabel: {'Positive' if y_train_raw[0] == 1 else 'Negative'}")
print(f"\nFull dataset: {len(X_train_raw):,} train, {len(X_test_raw):,} test")

In [ ]:
# Subsample for demo speed: 5,000 train, 1,000 test
np.random.seed(42)
train_idx = np.random.choice(len(X_train_raw), 5000, replace=False)
test_idx = np.random.choice(len(X_test_raw), 1000, replace=False)

X_train_seq = X_train_raw[train_idx]
y_train = y_train_raw[train_idx]
X_test_seq = X_test_raw[test_idx]
y_test = y_test_raw[test_idx]

# Pad/truncate to 200 tokens
MAX_LEN = 200
X_train_pad = np.array([seq[:MAX_LEN] if len(seq) > MAX_LEN
                         else seq + [0] * (MAX_LEN - len(seq))
                         for seq in X_train_seq])
X_test_pad = np.array([seq[:MAX_LEN] if len(seq) > MAX_LEN
                        else seq + [0] * (MAX_LEN - len(seq))
                        for seq in X_test_seq])

print(f"Training set: {X_train_pad.shape}")
print(f"Test set: {X_test_pad.shape}")
print(f"Class balance (train): {y_train.mean():.1%} positive")

### Build and Train an LSTM

A simple architecture: Embedding → LSTM → Dense → Sigmoid.

This is the same architecture from the slides, now with real data.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)              # (batch, seq_len, embed_dim)
        output, (hidden, cell) = self.lstm(embedded)  # output: (batch, seq_len, hidden_dim)
        hidden = hidden.squeeze(0)                 # (batch, hidden_dim)
        return torch.sigmoid(self.fc(hidden))      # (batch, 1)

VOCAB_SIZE = 10000
EMBED_DIM = 64
HIDDEN_DIM = 64

model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"  Embedding: {VOCAB_SIZE} x {EMBED_DIM} = {VOCAB_SIZE * EMBED_DIM:,}")
print(f"  LSTM: 4 x ({EMBED_DIM} + {HIDDEN_DIM} + 1) x {HIDDEN_DIM} = {4 * (EMBED_DIM + HIDDEN_DIM + 1) * HIDDEN_DIM:,}")
print(f"  Dense: {HIDDEN_DIM} x 1 + 1 = {HIDDEN_DIM + 1}")

In [ ]:
# Prepare data loaders
X_train_t = torch.tensor(X_train_pad, dtype=torch.long).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_t = torch.tensor(X_test_pad, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Training loop
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_losses = []
test_accs = []
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Evaluate
    model.eval()
    with torch.no_grad():
        test_out = model(X_test_t)
        test_preds = (test_out > 0.5).float()
        test_acc = (test_preds == y_test_t).float().mean().item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accs.append(test_acc)
    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}  Test Acc: {test_acc:.2%}")

print(f"\nFinal test accuracy: {test_accs[-1]:.2%}")

### Visualizing Hidden States

Now the interesting part: what does the LSTM's hidden state look like as it reads a sentence word by word?

Each column in the heatmap is one time step. Each row is one dimension of the hidden state. Watch how the pattern shifts when the model encounters sentiment-carrying words.

In [ ]:
def get_hidden_states(model, token_ids, device):
    """Extract hidden state at every time step."""
    model.eval()
    with torch.no_grad():
        x = torch.tensor([token_ids], dtype=torch.long).to(device)
        embedded = model.embedding(x)
        output, _ = model.lstm(embedded)  # output: (1, seq_len, hidden_dim)
    return output.squeeze(0).cpu().numpy()

def plot_hidden_states(hidden, words, title, ax):
    """Plot hidden state heatmap with word labels."""
    n_words = len(words)
    im = ax.imshow(hidden[:n_words].T, aspect='auto', cmap='RdBu_r',
                   vmin=-1, vmax=1)
    ax.set_xticks(range(n_words))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Hidden Dimension')
    ax.set_title(title, fontsize=12)
    return im

In [ ]:
# Encode two sentences: one positive, one negative
# Using our idx_to_word mapping in reverse
word_to_idx_imdb = {v: k for k, v in idx_to_word.items()}

def encode_sentence(sentence):
    """Encode a sentence using the IMDB vocabulary."""
    words = sentence.lower().split()
    ids = [word_to_idx_imdb.get(w, 2) for w in words]  # 2 = <UNK>
    return ids, words

pos_ids, pos_words = encode_sentence("this movie was absolutely wonderful and beautiful")
neg_ids, neg_words = encode_sentence("this movie was absolutely terrible and boring")

# Pad to MAX_LEN (model expects fixed length)
def pad_ids(ids, max_len=200):
    return ids + [0] * (max_len - len(ids))

pos_hidden = get_hidden_states(model, pad_ids(pos_ids), device)
neg_hidden = get_hidden_states(model, pad_ids(neg_ids), device)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

im0 = plot_hidden_states(pos_hidden, pos_words,
    'Hidden States: "this movie was absolutely wonderful and beautiful" (Positive)', axes[0])
plt.colorbar(im0, ax=axes[0])

im1 = plot_hidden_states(neg_hidden, neg_words,
    'Hidden States: "this movie was absolutely terrible and boring" (Negative)', axes[1])
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("Notice: the first three words are identical in both sentences.")
print("The hidden states start to diverge at 'wonderful' vs 'terrible'.")

### Order Matters: Hidden State Trajectories

RNNs process tokens sequentially, so word order changes the hidden state trajectory. Let's visualize this with PCA: project the hidden state at each time step down to 2D and trace the path.

In [ ]:
# Two sentences with the same words, different order
sent_a_ids, sent_a_words = encode_sentence("the movie was not good it was bad")
sent_b_ids, sent_b_words = encode_sentence("the movie was not bad it was good")

hidden_a = get_hidden_states(model, pad_ids(sent_a_ids), device)
hidden_b = get_hidden_states(model, pad_ids(sent_b_ids), device)

# Trim to actual sentence length
hidden_a = hidden_a[:len(sent_a_words)]
hidden_b = hidden_b[:len(sent_b_words)]

# PCA on combined hidden states
from sklearn.decomposition import PCA
combined = np.vstack([hidden_a, hidden_b])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(combined)

coords_a = coords[:len(sent_a_words)]
coords_b = coords[len(sent_a_words):]

fig, ax = plt.subplots(figsize=(12, 8))

# Plot trajectories
ax.plot(coords_a[:, 0], coords_a[:, 1], 'o-', color='coral',
        linewidth=2, markersize=8, label='"...not good...was bad"')
ax.plot(coords_b[:, 0], coords_b[:, 1], 's-', color='steelblue',
        linewidth=2, markersize=8, label='"...not bad...was good"')

# Label each time step
for i, word in enumerate(sent_a_words):
    ax.annotate(f'{word} (t={i})', (coords_a[i, 0], coords_a[i, 1]),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points', color='coral')
for i, word in enumerate(sent_b_words):
    ax.annotate(f'{word} (t={i})', (coords_b[i, 0], coords_b[i, 1]),
                fontsize=9, ha='center', va='top',
                xytext=(0, -10), textcoords='offset points', color='steelblue')

ax.set_title('Hidden State Trajectories: Same Words, Different Order', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Same words, different order -> different hidden state trajectories.")
print("The final hidden state (used for classification) ends up in a different place.")

**Takeaways from Demo 2:**
- The LSTM hidden state evolves at each time step, accumulating information
- Sentiment-carrying words ("wonderful" vs "terrible") cause visible shifts in the hidden state
- Word order matters: the same words in different arrangements produce different trajectories
- The final hidden state summarizes the entire sequence into a fixed-size vector for classification

---
# Demo 3: Attention Visualization

**Goal:** See self-attention weights in a pretrained BERT model.

This makes the Query/Key/Value mechanism tangible: for each token, we can see exactly which other tokens it attends to.

In [ ]:
# Install transformers (takes ~30 seconds)
!pip install transformers -q

from transformers import BertTokenizer, BertModel
import torch

print("Transformers installed successfully!")

In [ ]:
# Load pretrained BERT (downloads ~440MB on first run, ~30 seconds)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)
bert_model.eval()

print(f"Model loaded: bert-base-uncased")
print(f"Hidden size: {bert_model.config.hidden_size}")
print(f"Attention heads: {bert_model.config.num_attention_heads}")
print(f"Layers: {bert_model.config.num_hidden_layers}")

### Self-Attention: What Does "it" Refer To?

The classic test: "The cat sat on the mat because **it** was tired."

A good language model should learn that "it" refers to "cat", not "mat". Let's see if BERT's attention agrees.

In [ ]:
def get_attention(model, tokenizer, sentence):
    """Get attention weights from BERT for a sentence."""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model(**inputs)

    # outputs.attentions: tuple of (batch, heads, seq_len, seq_len) per layer
    # Stack all layers: (layers, heads, seq_len, seq_len)
    attention = torch.stack(outputs.attentions).squeeze(1)
    return attention.numpy(), tokens

def plot_attention_head(attention, tokens, layer, head, ax, title=None):
    """Plot attention weights for one head as a heatmap."""
    attn = attention[layer, head]
    n = len(tokens)
    im = ax.imshow(attn[:n, :n], cmap='Blues', vmin=0)
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n))
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel('Attended To (Keys)')
    ax.set_ylabel('Attending From (Queries)')
    if title:
        ax.set_title(title, fontsize=11)
    return im

print("Visualization functions ready.")

In [ ]:
# The pronoun resolution test
sentence = "The cat sat on the mat because it was tired."
attention, tokens = get_attention(bert_model, tokenizer, sentence)

print(f"Tokens: {tokens}")
print(f"Attention shape: {attention.shape}")
print(f"  {attention.shape[0]} layers x {attention.shape[1]} heads x {attention.shape[2]} tokens x {attention.shape[3]} tokens")

In [ ]:
# Find what "it" attends to across different heads
it_idx = tokens.index('it')
print(f"Token 'it' is at position {it_idx}\n")

# Show attention from "it" across several heads in layer 5
# (mid-network layers tend to show the most interpretable syntactic patterns)
layer = 5

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for head in range(8):
    attn_weights = attention[layer, head, it_idx, :len(tokens)]
    ax = axes[head]
    bars = ax.bar(range(len(tokens)), attn_weights, color='steelblue', alpha=0.8)

    # Highlight the "cat" bar
    cat_idx = tokens.index('cat')
    if attn_weights[cat_idx] > 0.05:
        bars[cat_idx].set_color('coral')
        bars[cat_idx].set_alpha(1.0)

    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Layer {layer}, Head {head}', fontsize=10)
    ax.set_ylim(0, 1)

plt.suptitle(f'What does "it" attend to? (Layer {layer}, all 8 heads shown)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each head learns a different attention pattern.")
print("Some heads connect 'it' to 'cat' (coreference resolution).")
print("Other heads attend to nearby words or punctuation (positional patterns).")

In [ ]:
# Full attention heatmap for one head
# Pick a head where "it" attends to "cat" (find it programmatically)
cat_idx = tokens.index('cat')
it_idx = tokens.index('it')

# Find the head in layer 5 that attends most to "cat" from "it"
best_head = 0
best_score = 0
for h in range(attention.shape[1]):
    score = attention[layer, h, it_idx, cat_idx]
    if score > best_score:
        best_score = score
        best_head = h

fig, ax = plt.subplots(figsize=(10, 8))
im = plot_attention_head(attention, tokens, layer=layer, head=best_head, ax=ax,
    title=f'Self-Attention Heatmap (Layer {layer}, Head {best_head})')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print(f"\nHead {best_head} in layer {layer}: 'it' attends to 'cat' with weight {best_score:.3f}")
print("Read the heatmap: row = query (attending FROM), column = key (attending TO)")
print("Bright squares show strong attention connections.")

### Comparing Attention Across Sentences

Let's see how attention changes when we swap the structure.

In [ ]:
# Compare: change what "it" should refer to
sentences = [
    "The cat sat on the mat because it was tired.",
    "The cat sat on the mat because it was dirty.",
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for i, sent in enumerate(sentences):
    attn, toks = get_attention(bert_model, tokenizer, sent)
    it_pos = toks.index('it')

    # Same layer/head as before for consistency
    attn_from_it = attn[layer, best_head, it_pos, :len(toks)]

    bars = axes[i].bar(range(len(toks)), attn_from_it, color='steelblue', alpha=0.8)

    # Highlight top 2 attended tokens (excluding special tokens and "it" itself)
    scores = [(j, attn_from_it[j]) for j in range(len(toks))
              if toks[j] not in ['[CLS]', '[SEP]', 'it']]
    scores.sort(key=lambda x: x[1], reverse=True)
    for j, _ in scores[:2]:
        bars[j].set_color('coral')
        bars[j].set_alpha(1.0)

    axes[i].set_xticks(range(len(toks)))
    axes[i].set_xticklabels(toks, rotation=45, ha='right', fontsize=10)
    axes[i].set_title(f'"{sent}"', fontsize=11)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Attention Weight')

plt.suptitle(f'What does "it" attend to? (Layer {layer}, Head {best_head})',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("'it was tired' -> 'it' should attend to 'cat' (cats get tired)")
print("'it was dirty' -> 'it' could attend to 'mat' (mats get dirty)")
print("Different context, different attention pattern. This is the power of attention.")

**Takeaways from Demo 3:**
- Self-attention lets every token directly attend to every other token (no sequential bottleneck)
- Different attention heads learn different patterns (syntax, coreference, position)
- Attention is context-dependent: "it" attends to different words depending on the rest of the sentence
- This is what makes transformers more powerful than RNNs for long-range dependencies

---
# Summary

**Demo 1 (Embeddings):** Words become vectors. Similar words are nearby. Arithmetic on vectors captures relationships.

**Demo 2 (RNN Hidden States):** The hidden state evolves at each time step, building up a summary of the sequence. Order matters; the same words in different order produce different representations.

**Demo 3 (Attention):** Self-attention connects every token to every other token directly. Different heads learn different patterns. Context determines where the model focuses.

**The progression:** Embeddings solve the "words to numbers" problem. RNNs solve the "sequences have order" problem. Attention solves the "long-range dependencies" and "parallelization" problems. Transformers combine attention with feed-forward networks into the architecture behind modern LLMs.